# 06 — Social charts (Pillow)

The publication-ready charts, each rendered **inline** (then saved to
`outputs/social/`). Workspace rule: `display()` first, then save — a chart is
never saved without being shown.

1. Share of box office earned home vs abroad (100% stacked bar)
2. Biggest *domestic* films, adjusted for inflation (dumbbell)

*(The genre 'share earned abroad' chart stays exploration-only in `04-viz` —
the genres are all too close in value to make a compelling social chart. A
third social chart — international films with the largest U.S. box office — is
planned; see `artifacts/PROJECT-STATUS.md`.)*

In [ ]:
import sys, os
from pathlib import Path
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

WORKSPACE = PROJECT.parent.parent
sys.path.insert(0, str(WORKSPACE / 'shared'))

import duckdb
from src.ingest import load_config
from chart_templates import lollipop, stacked_100pct_bars
from viz import PRESETS
from IPython.display import display

cfg = load_config('config.yaml')
# read-only: viz notebooks only read, so they run alongside an open kernel.
con = duckdb.connect(cfg['settings']['duckdb_file'], read_only=True)
img_w, img_h, _ = PRESETS['twitter_landscape']
out = Path(cfg['paths']['outputs_social']); out.mkdir(parents=True, exist_ok=True)
def money(v):
    return f'${v/1e9:.2f}B' if abs(v) >= 1e9 else f'${v/1e6:.0f}M'

## 1. Where the money comes from: overseas
Top 15 by worldwide gross. Gold = domestic (US/Canada), teal = international.

In [ ]:
# 100% stacked bar: each film's split home vs abroad, ordered by share abroad
# (the teal segment shrinks down the list, so the visual matches the sort).
ww = con.execute('''
    WITH us AS (SELECT title, release_year FROM films_genre WHERE is_us=TRUE),
    top AS (SELECT w.title, w.release_year,
                   100.0*w.foreign_gross/w.worldwide_gross AS foreign_pct,
                   100.0*w.domestic_gross/w.worldwide_gross AS home_pct
            FROM films_worldwide w JOIN us ON us.title=w.title AND us.release_year=w.release_year
            ORDER BY w.worldwide_gross DESC LIMIT 15)
    SELECT * FROM top ORDER BY foreign_pct DESC''').df()
ww['label'] = ww['title'] + '  (' + ww['release_year'].astype(str) + ')'
img2 = stacked_100pct_bars(ww, group_col='label',
    segments=[{'col':'foreign_pct','label':'Rest of world','color':'#005F73'},
              {'col':'home_pct','label':'Home (US/Canada)','color':'#EE9B00'}],
    title='Hollywood films: share of box office earned at home vs abroad',
    subtitle='Top 15 U.S.-produced films by worldwide gross, ordered by share earned abroad.',
    source='Box Office Mojo, Top Lifetime Grosses (Worldwide) + TMDB origin country — as of Sep 2026',
    bar_height=34, bar_gap=12, img_width=img_w, img_height=img_h)
display(img2)
img2.save(out / '02_worldwide_domestic_vs_international.png')

## 2. Biggest *domestic* films, adjusted for inflation
A different question: within the U.S. & Canada, adjusted for ticket-price
inflation. Teal = adjusted (2022 $), gold = nominal (release $).

In [ ]:
dom = con.execute('''SELECT title, adjusted_gross, nominal_gross, release_year
    FROM films_adjusted ORDER BY adjusted_gross DESC LIMIT 15''').df()
dom['label'] = dom['title'] + '  (' + dom['release_year'].astype(str) + ')'
img1 = lollipop(dom, category_col='label', value_col='adjusted_gross', value2_col='nominal_gross',
    value_fmt=money, title='The all-time top 15 domestic films, adjusted for inflation',
    subtitle='Ranked by inflation-adjusted U.S. & Canada gross. Teal = adjusted to 2022 $, gold = what each film actually made at the time (nominal).',
    source='Box Office Mojo, Top Lifetime Adjusted Grosses (domestic, adj. to 2022) — as of Sep 2026',
    dot_color='#005F73', dot2_color='#EE9B00',
    value_label='Adjusted (2022 $)', value2_label='Nominal (release $)',
    img_width=img_w, img_height=img_h)
display(img1)
img1.save(out / '01_domestic_adjusted_vs_nominal.png')

---
Three charts written to `outputs/social/`. Watermark `@unwelcomedata`, brand
palette, `twitter_landscape` preset.

## Cleanup
Close the DuckDB connection so the lock is released for other tools.

In [ ]:
con.close()
print('connection closed')